# 11 — Real-Time Endpoint Validation

## Purpose

This notebook continues directly after `10_model_registry.ipynb`. Notebook 10 registered the selected XGBoost model and validated batch inference on production-simulation data. This notebook adds the next operational proof point: a SageMaker real-time endpoint that can return a delay probability for a single flight record.

The goal is to validate that the selected model can support an interactive stakeholder workflow, such as a flight operations dashboard or customer-service decision support tool, without repeating earlier S3, Athena, Feature Store, training, or registry work.

## Prerequisites

Run notebooks `01_s3_setup.ipynb` through `10_model_registry.ipynb` first. This notebook expects the following stored variables from prior notebooks: `s3_aerodelay`, `best_model_uri`, `model_package_arn`, `model_package_group`, and `batch_output_s3`.

## Cost Control Note

The endpoint deployment cell is guarded by `DEPLOY_ENDPOINT = False` by default. Change it to `True` only when you are ready to validate real-time inference for the screencast, and run the cleanup cell at the end when the endpoint is no longer needed.


## 1. Setup

This step recreates the same AWS and SageMaker clients used in the earlier modular notebooks. It also creates a local `reports/` folder for validation artifacts that can be committed to GitHub or copied into the final project report.


In [ ]:
import boto3, os, json, time
import pandas as pd
import awswrangler as wr
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
bucket = sess.default_bucket()
role = get_execution_role()
region = sess.boto_region_name

sm = boto3.client("sagemaker", region_name=region)
s3 = boto3.client("s3", region_name=region)
runtime = boto3.client("sagemaker-runtime", region_name=region)

os.makedirs("reports", exist_ok=True)

print(f"Bucket : {bucket}")
print(f"Region : {region}")


In [ ]:
%store -r s3_aerodelay
%store -r best_model_uri
%store -r model_package_arn
%store -r model_package_group
%store -r batch_output_s3

print(f"s3_aerodelay       : {s3_aerodelay}")
print(f"best_model_uri     : {best_model_uri}")
print(f"model_package_arn  : {model_package_arn}")
print(f"model_package_group: {model_package_group}")
print(f"batch_output_s3    : {batch_output_s3}")


## 2. Verify Registered Model and Model Artifact

Before creating an endpoint, this step verifies that the selected model package exists and that the model artifact produced in notebook 10 is available in S3. The clean model artifact is preferred because notebook 10 prepared it specifically for SageMaker XGBoost batch and real-time inference.


In [ ]:
clean_model_uri = f"{s3_aerodelay}/model-artifacts/clean/xgboost-model.tar.gz"

def s3_uri_exists(s3_uri):
    bucket_name = s3_uri.split("/")[2]
    key = "/".join(s3_uri.split("/")[3:])
    try:
        response = s3.head_object(Bucket=bucket_name, Key=key)
        return True, response["ContentLength"]
    except Exception as e:
        return False, str(e)

artifact_exists, artifact_detail = s3_uri_exists(clean_model_uri)
if artifact_exists:
    model_artifact_uri = clean_model_uri
    print(f"Clean model artifact verified: {model_artifact_uri} ({artifact_detail/1024:.0f} KB)")
else:
    model_artifact_uri = best_model_uri
    print("Clean model artifact not found; falling back to best_model_uri.")
    print(f"Fallback model artifact: {model_artifact_uri}")

package_desc = sm.describe_model_package(ModelPackageName=model_package_arn)
print(f"Model package status : {package_desc['ModelPackageStatus']}")
print(f"Approval status      : {package_desc.get('ModelApprovalStatus', 'N/A')}")


## 3. Create or Reuse the SageMaker Model

This step creates the SageMaker model object used by the endpoint. It is idempotent: if the model already exists, the notebook reuses it instead of creating a duplicate resource.


In [ ]:
from sagemaker.core.image_uris import retrieve

model_name = "aerodelay-xgb-realtime-model"
xgb_image_uri = retrieve(framework="xgboost", region=region, version="1.7-1")

try:
    sm.describe_model(ModelName=model_name)
    print(f"Model already exists: {model_name}")
except Exception:
    sm.create_model(
        ModelName=model_name,
        ExecutionRoleArn=role,
        PrimaryContainer={
            "Image": xgb_image_uri,
            "ModelDataUrl": model_artifact_uri,
        },
    )
    print(f"Created model: {model_name}")

%store model_name
print(f"Model image       : {xgb_image_uri}")
print(f"Model artifact    : {model_artifact_uri}")


## 4. Create Endpoint Configuration

This cell defines the compute configuration for real-time inference. The default instance type is `ml.m5.large`, which is appropriate for a short demonstration in the Learner Lab environment. The cell reuses an existing endpoint configuration if one already exists.


In [ ]:
endpoint_config_name = "aerodelay-xgb-realtime-config"
endpoint_name = "aerodelay-xgb-realtime-endpoint"

try:
    sm.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"Endpoint config already exists: {endpoint_config_name}")
except Exception:
    sm.create_endpoint_config(
        EndpointConfigName=endpoint_config_name,
        ProductionVariants=[
            {
                "VariantName": "AllTraffic",
                "ModelName": model_name,
                "InitialInstanceCount": 1,
                "InstanceType": "ml.m5.large",
                "InitialVariantWeight": 1.0,
            }
        ],
    )
    print(f"Created endpoint config: {endpoint_config_name}")

%store endpoint_config_name
%store endpoint_name


## 5. Deploy or Restore Real-Time Endpoint

Set `DEPLOY_ENDPOINT = True` only when you are ready to create or update the endpoint. Leaving the flag as `False` makes the notebook safe to commit and review without accidentally creating a billable resource.


In [ ]:
DEPLOY_ENDPOINT = False  # Change to True only when ready to run the live endpoint demo.

endpoint_exists = False
try:
    endpoint_desc = sm.describe_endpoint(EndpointName=endpoint_name)
    endpoint_exists = True
    print(f"Endpoint already exists: {endpoint_name}")
    print(f"Current status: {endpoint_desc['EndpointStatus']}")
except Exception:
    print(f"Endpoint does not exist yet: {endpoint_name}")

if DEPLOY_ENDPOINT and not endpoint_exists:
    sm.create_endpoint(
        EndpointName=endpoint_name,
        EndpointConfigName=endpoint_config_name,
    )
    print(f"Endpoint creation started: {endpoint_name}")
elif not DEPLOY_ENDPOINT and not endpoint_exists:
    print("Deployment skipped because DEPLOY_ENDPOINT is False.")
    print("For the screencast, set DEPLOY_ENDPOINT = True and rerun this cell.")


## 6. Wait for Endpoint to Become InService

This cell waits only if an endpoint exists. It is safe to run after deployment or when restoring an already-created endpoint.


In [ ]:
try:
    while True:
        desc = sm.describe_endpoint(EndpointName=endpoint_name)
        status = desc["EndpointStatus"]
        print(f"Endpoint status: {status}")

        if status == "InService":
            break
        if status in ["Failed", "OutOfService"]:
            raise RuntimeError(desc.get("FailureReason", "Endpoint failed without a detailed reason."))

        time.sleep(30)
except Exception as e:
    print(f"Endpoint wait skipped or unavailable: {e}")


## 7. Prepare One Production-Simulation Record

This step loads one row from the production-simulation split, drops the target label, encodes string features in the same simple way used during evaluation, and converts the row to CSV for the SageMaker XGBoost container.


In [ ]:
from sklearn.preprocessing import LabelEncoder

def encode_strings(df):
    df = df.copy()
    for col in df.select_dtypes(include=["object", "string"]).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    return df

prod_df = wr.s3.read_parquet(f"{s3_aerodelay}/production_simulation/data.parquet")
sample_df = prod_df.head(1).copy()
actual_delay = int(sample_df["arrdel15"].iloc[0]) if "arrdel15" in sample_df.columns else None
feature_df = sample_df.drop(columns=["arrdel15"], errors="ignore")
feature_df_encoded = encode_strings(feature_df).fillna(0)

payload = feature_df_encoded.to_csv(header=False, index=False).strip()

print(f"Sample feature count : {feature_df_encoded.shape[1]}")
print(f"Actual delay label   : {actual_delay}")
print(f"Payload preview      : {payload[:160]}...")


## 8. Invoke Endpoint and Save Validation Result

This step performs the real-time inference call. If the endpoint is not deployed, the cell prints a clear message and creates a placeholder validation record explaining what still needs to be run for the screencast.


In [ ]:
validation_result = {
    "endpoint_name": endpoint_name,
    "endpoint_available": False,
    "delay_probability": None,
    "predicted_delay": None,
    "actual_delay": actual_delay,
    "status": "not_invoked",
}

try:
    endpoint_desc = sm.describe_endpoint(EndpointName=endpoint_name)
    if endpoint_desc["EndpointStatus"] == "InService":
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType="text/csv",
            Body=payload,
        )
        raw_prediction = response["Body"].read().decode("utf-8").strip()
        delay_probability = float(raw_prediction.split(",")[0])
        predicted_delay = int(delay_probability >= 0.5)

        validation_result.update({
            "endpoint_available": True,
            "delay_probability": delay_probability,
            "predicted_delay": predicted_delay,
            "status": "invoked_successfully",
        })

        print("Endpoint invocation succeeded.")
        print(f"Delay probability : {delay_probability:.4f}")
        print(f"Predicted delay   : {predicted_delay}")
        print(f"Actual delay      : {actual_delay}")
    else:
        print(f"Endpoint exists but is not InService: {endpoint_desc['EndpointStatus']}")
except Exception as e:
    validation_result["status"] = f"not_invoked: {e}"
    print(f"Endpoint invocation skipped: {e}")

with open("reports/realtime_endpoint_validation.json", "w") as f:
    json.dump(validation_result, f, indent=2)

wr.s3.to_json(
    df=pd.DataFrame([validation_result]),
    path=f"{s3_aerodelay}/reports/realtime_endpoint_validation.json",
)

realtime_validation_s3 = f"{s3_aerodelay}/reports/realtime_endpoint_validation.json"
%store realtime_validation_s3
print(f"Saved validation result locally and to: {realtime_validation_s3}")


## 9. Optional Cleanup Cell

Use this cell after the screencast if the endpoint was deployed only for demonstration. The default flag prevents accidental deletion.


In [ ]:
DELETE_ENDPOINT = False  # Change to True only after the real-time endpoint demo is complete.

if DELETE_ENDPOINT:
    try:
        sm.delete_endpoint(EndpointName=endpoint_name)
        print(f"Deleted endpoint: {endpoint_name}")
    except Exception as e:
        print(f"Endpoint delete skipped: {e}")
else:
    print("Cleanup skipped. Set DELETE_ENDPOINT = True when you are ready to remove the live endpoint.")


## 10. Summary — Real-Time Endpoint Validation

This notebook extends the existing ML system by validating a real-time inference path after the model has already been trained, tuned, evaluated, registered, and used for batch inference. The notebook is safe for GitHub because endpoint creation and deletion are controlled by explicit Boolean flags.

| Deliverable | Status | Location |
|---|---|---|
| SageMaker model object | Created or reused | `aerodelay-xgb-realtime-model` |
| Endpoint configuration | Created or reused | `aerodelay-xgb-realtime-config` |
| Real-time endpoint | Guarded deployment | `aerodelay-xgb-realtime-endpoint` |
| Invocation validation artifact | Saved locally and to S3 | `reports/realtime_endpoint_validation.json` |
| Downstream variable | Stored | `realtime_validation_s3` |

For the final demonstration, show the endpoint status, invoke one production-simulation record, explain the delay probability, and then confirm that cleanup is available to avoid leaving unnecessary resources running.
